# 07c — Historical External Context Source Audit & Alignment

Gate-first stage: assign temporally appropriate historical sources to recurring non-HOME anchors before any semantic enrichment.

No historical POI dataset is downloaded in this notebook.


In [ ]:
from pathlib import Path
import subprocess, sys
REPO_URL='https://github.com/tanh1c/geolife.git'
BRANCH='eda/07e-semantic-distance-alignment'
REPO_DIR=Path('/root/geolife')
if not (REPO_DIR/'.git').exists():
    subprocess.run(['git','clone','--branch',BRANCH,'--single-branch',REPO_URL,str(REPO_DIR)],check=True)
else:
    subprocess.run(['git','-C',str(REPO_DIR),'fetch','origin',BRANCH],check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'checkout',BRANCH],check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'reset','--hard',f'origin/{BRANCH}'],check=True)
print('sha:',subprocess.run(['git','-C',str(REPO_DIR),'rev-parse','--short','HEAD'],capture_output=True,text=True,check=True).stdout.strip())

# Fresh Modal runtimes need the src-layout package installed before loading analysis helpers.
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.[dev]','timezonefinder==9.0.0'],cwd=REPO_DIR,check=True)
repo_root=str(REPO_DIR); src_root=str(REPO_DIR/'src')
if repo_root not in sys.path: sys.path.insert(0,repo_root)
if src_root not in sys.path: sys.path.insert(0,src_root)
print('editable package installed')


In [ ]:
from importlib.util import spec_from_file_location,module_from_spec
import pandas as pd
from geolife.model import HomeOfficeConfig, build_semantic_locations
def load(name,path):
    spec=spec_from_file_location(name,path); m=module_from_spec(spec); sys.modules[name]=m; spec.loader.exec_module(m); return m
s03=load('s03_07c',REPO_DIR/'analysis'/'03a_user_behavior_deep_dive.py')
s07b=load('s07b_07c',REPO_DIR/'analysis'/'07b_factorized_work_profiles.py')
s07c=load('s07c_hist',REPO_DIR/'analysis'/'07c_historical_source_alignment.py')
print(s07c.synthetic_self_check())


In [ ]:
V=Path('/mnt/geolife-data'); C=V/'cache'/'07c_historical_source_alignment'; C.mkdir(parents=True,exist_ok=True)
def find(name):
    xs=sorted(V.glob(f'**/{name}'))
    if not xs: raise FileNotFoundError(name)
    return xs[0]
def optional(name):
    xs=sorted(V.glob(f'**/{name}')); return xs[0] if xs else None
STAYS=find('stays_baseline_v1.pkl'); HOME=find('home_consensus_private.pkl'); FEATURES=find('user_behavior_features.csv')
WORK=find('adaptive_work_patterns_42_private.pkl'); ROUTINE=find('user_summary_private.pkl'); INDEP=optional('candidate_comparison_private.pkl'); PROFILE=optional('factorized_work_profiles_private.pkl')
print('private inputs located')


In [ ]:
stays=pd.read_pickle(STAYS); home=pd.read_pickle(HOME); work=pd.read_pickle(WORK); routine=pd.read_pickle(ROUTINE)
indep=pd.read_pickle(INDEP) if INDEP else None
cfg=HomeOfficeConfig(clustering_method='complete_link',location_max_diameter_m=200.0)
semantic_stays,locations=build_semantic_locations(stays,config=cfg)
if PROFILE:
    profiles=pd.read_pickle(PROFILE)
else:
    features=pd.read_csv(FEATURES,dtype={'user_id':str})
    if not {'stable_shifted_candidate','mobile_work_like_candidate'}.issubset(features.columns): features=s03.assign_behavioral_candidates(features)
    profiles=s07b.run_audit(features,work,routine,home_evidence=home,independent_secondary_evidence=indep).profiles
namespace_validation=s07c.validate_location_namespace(locations,home,work)
display(namespace_validation)
anchors=s07c.build_support_qualified_anchors(locations,home,profiles,min_stay_count=2)
anchors['location_namespace']=s07c.PRODUCTION_LOCATION_NAMESPACE
dated=s07c.summarize_anchor_observation_dates(semantic_stays,anchors)
plan=s07c.build_temporal_source_plan(dated)
print('candidate users:',anchors.user_id.nunique(),'candidate anchors:',len(anchors))


In [ ]:
print('SOURCE GATES'); display(s07c.source_gate_summary())
print('ANCHOR OBSERVATION YEARS'); display(dated.groupby('median_observation_year',as_index=False).agg(anchors=('location_id','size'),users=('user_id','nunique')).sort_values('median_observation_year'))
print('SOURCE PLAN SUMMARY'); display(s07c.summarize_plan(plan))
print('SOURCE YEAR COVERAGE'); display(s07c.summarize_year_coverage(plan))


In [ ]:
dated.to_pickle(C/'anchor_observation_dates_private.pkl')
plan.to_pickle(C/'anchor_source_plan_private.pkl')
namespace_validation.to_csv(C/'location_namespace_validation.csv',index=False)
s07c.source_gate_summary().to_csv(C/'source_gate_summary.csv',index=False)
s07c.summarize_plan(plan).to_csv(C/'source_plan_summary.csv',index=False)
s07c.summarize_year_coverage(plan).to_csv(C/'source_year_coverage.csv',index=False)
dated.groupby('median_observation_year',as_index=False).agg(anchors=('location_id','size'),users=('user_id','nunique')).to_csv(C/'anchor_year_summary.csv',index=False)
print('saved:',C)


## Interpretation boundary

- `blocked_*` sources must not be loaded automatically by Stage 07d.
- nearest-year proxy is not exact historical truth.
- CLCD is physical land-cover context only.
- historical OSM is a cross-check; missing OSM does not prove absence.
